# CEO Dashboard — MLflow LLM-as-Judge Evaluation

This notebook demonstrates how to evaluate the Casper's Kitchens CEO Supervisor using **MLflow GenAI evaluation**. It covers:

1. **Evaluation datasets** — curated questions with behavioral guidelines (data-agnostic, so they survive simulator updates)
2. **LLM-as-judge scorers** — built-in `Guidelines` and `Safety` judges + custom `routing_accuracy` and `cites_specific_data` scorers
3. **Running evaluation** — `mlflow.genai.evaluate()` against the live MAS endpoint
4. **Trace-derived datasets** — how to harvest production traces and turn them into a labelled eval set

The CEO Supervisor routes questions across 7 sub-agents:
```
CEO Supervisor (MAS)
  ├── Revenue Analytics Genie    → lakeflow + simulator Delta tables
  ├── Operations Intelligence Genie → food_safety + all_events tables
  ├── Inspection Reports KA      → food safety PDF reports
  ├── Legal Complaints KA        → litigation case file PDFs
  ├── Regulatory Documents KA    → permits & certifications PDFs
  ├── Audit Findings KA          → financial / operational audit PDFs
  └── Consultancy Strategy KA    → management consulting report PDFs
```

> **Demo tip:** Run cells 1–5 before the live demo. Results will be ready to show in the MLflow Experiments UI during Part 3 of the demo script.

In [ ]:
%pip install --upgrade mlflow databricks-sdk 'mlflow[databricks]'
dbutils.library.restartPython()

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
import sys
sys.path.append("../../utils")

from databricks.sdk import WorkspaceClient

try:
    CATALOG = dbutils.widgets.get("CATALOG")
except Exception:
    CATALOG = "caspersdev"

_w = WorkspaceClient()
_supervisor_tile_name = f"{CATALOG}-ceo-supervisor"
_current_user = spark.sql("SELECT current_user()").collect()[0][0]


def _resolve_supervisor_endpoint(tile_name: str) -> str:
    """Return the serving endpoint name for the named MAS tile."""
    # Primary: MAS API
    try:
        params = {}
        while True:
            resp = _w.api_client.do("GET", "/api/2.0/multi-agent-supervisors", query=params)
            for item in resp.get("multi_agent_supervisors", []):
                mas = item.get("multi_agent_supervisor", item)
                if mas.get("name") == tile_name:
                    ep = mas.get("endpoint_name", "")
                    if ep:
                        return ep
            token = resp.get("next_page_token")
            if not token:
                break
            params = {"page_token": token}
    except Exception as e:
        print(f"⚠️  MAS API lookup failed: {e}")

    # Fallback: generic tiles API
    try:
        params = {}
        while True:
            resp = _w.api_client.do("GET", "/api/2.0/tiles", query=params)
            for tile in resp.get("tiles", []):
                if tile.get("name") == tile_name:
                    ep = tile.get("serving_endpoint_name", "")
                    if ep:
                        return ep
            token = resp.get("next_page_token")
            if not token:
                break
            params = {"page_token": token}
    except Exception as e:
        print(f"⚠️  Tiles API fallback failed: {e}")

    return ""


SUPERVISOR_ENDPOINT = _resolve_supervisor_endpoint(_supervisor_tile_name)
if not SUPERVISOR_ENDPOINT:
    raise ValueError(
        f"Could not resolve supervisor endpoint for tile '{_supervisor_tile_name}'. "
        "Make sure the CEO Supervisor stage has been deployed for this catalog."
    )

print(f"Catalog:             {CATALOG}")
print(f"Supervisor endpoint: {SUPERVISOR_ENDPOINT}")


In [ ]:
import mlflow
import mlflow.genai
import pandas as pd
import json

# Derive the stable experiment name from the endpoint name.
# Pattern: mas-{tile_id_prefix}-dev-experiment
# "-dev-experiment" is the hardcoded suffix used across all eval runs.
_mas_id = SUPERVISOR_ENDPOINT.replace("-endpoint", "")
EVAL_EXPERIMENT_NAME = f"/Users/{_current_user}/{_mas_id}-dev-experiment"

mlflow.set_experiment(EVAL_EXPERIMENT_NAME)
print(f"Eval experiment: {EVAL_EXPERIMENT_NAME}")

w = _w
print(f"MLflow version: {mlflow.__version__}")


## 1. Define the `predict_fn`

The predict function wraps the CEO Supervisor endpoint. MLflow calls it for each row in the evaluation dataset, unpacking the `inputs` dict as keyword arguments.

The supervisor is a Multi-Agent Supervisor served at a Databricks Inference Endpoint — we query it exactly like any other chat model.

In [ ]:
from mlflow.deployments import get_deploy_client as _get_deploy_client

# mlflow.deployments client is the correct way to call endpoints inside
# mlflow.genai.evaluate() — it sends plain dicts without the SDK's
# object-serialisation layer that causes "'dict' has no attribute 'as_dict'".
_deploy_client = _get_deploy_client("databricks")


def _extract_text(response: dict) -> str:
    """
    Extract the assistant text from any known MAS response format:
    - Chat Completions: {"choices": [{"message": {"content": "..."}}]}
    - Responses API:    {"output": [{"role": "assistant", "content": "..." | [...]}]}
    - Agent final:      {"final_response": "..."}
    - Plain string:     "..."
    """
    # Deploy client sometimes returns a JSON string instead of a parsed dict —
    # try to parse it before treating it as plain text.
    if isinstance(response, str):
        try:
            response = json.loads(response)
        except (json.JSONDecodeError, ValueError):
            return response  # genuinely plain text

    if not isinstance(response, dict):
        return str(response)

    # {"final_response": "..."}
    if "final_response" in response:
        return str(response["final_response"])

    # Deploy client may wrap in {"predictions": [...] or {...}}
    preds = response.get("predictions")
    if preds is not None:
        if isinstance(preds, list) and preds:
            return _extract_text(preds[0])
        if isinstance(preds, (dict, str)):
            return _extract_text(preds)

    # Chat Completions format
    if "choices" in response:
        return response["choices"][0]["message"]["content"]

    # Responses API format
    output = response.get("output", [])
    if isinstance(output, list):
        for item in output:
            if not isinstance(item, dict):
                continue
            if item.get("role") == "assistant" or item.get("type") == "message":
                content = item.get("content", "")
                if isinstance(content, str):
                    return content
                if isinstance(content, list):
                    return " ".join(
                        c.get("text", c.get("value", ""))
                        for c in content
                        if isinstance(c, dict)
                    )
    if isinstance(output, str):
        return output

    print(f"\u26a0\ufe0f  Unrecognised response format \u2014 keys: {list(response.keys())}. Returning raw string.")
    return str(response)


def predict_fn(question: str, expected_agent: str = None) -> str:
    """
    Call the CEO Supervisor MAS and return its text response.

    mlflow.genai.evaluate() calls this for each dataset row, unpacking the
    `inputs` dict as kwargs.  MAS endpoints use the agent-style `input` field
    and return either Chat Completions or Responses API format depending on
    the SDK version — _extract_text handles both.

    Per-row exceptions are caught and returned as an error string so a single
    failed call does not abort the entire evaluation run.
    """
    try:
        response = _deploy_client.predict(
            endpoint=SUPERVISOR_ENDPOINT,
            inputs={"input": [{"role": "user", "content": question}]},
        )
        return _extract_text(response)
    except Exception as e:
        print(f"⚠️  predict_fn error for question {question[:60]!r}: {e}")
        return f"[ERROR: {e}]"


print(f"✅ predict_fn defined — will call {SUPERVISOR_ENDPOINT}")

## 2. Build the Evaluation Dataset

A good evaluation dataset for a multi-agent supervisor tests **agent behavior**, not specific data values.

Data changes — the simulator updates, new locations get added, legal cases resolve. Hard-coding "Chicago has the highest cancellation rate at 11-12%" would make the eval brittle and wrong the moment the data shifts.

Instead, each record defines **behavioral guidelines**: criteria that should hold true regardless of what the current data says:
- Did the agent route to the right sub-agent domain?
- Did it return a structured, data-backed response (numbers, dates, IDs)?
- Did it avoid vague hedging ("I cannot access...")?
- Did it follow the expected response format for that question type?

This also means we drop the `Correctness` scorer — that scorer compares outputs against a hardcoded `expected_response`, which is only appropriate when you have ground truth that doesn't change (like a fixed Q&A FAQ, not a live analytics system).

In [ ]:
# Dataset structure:
#   inputs  → kwargs passed to predict_fn; includes expected_agent so routing_accuracy
#             can read it without relying on the expectations kwarg (not forwarded by all
#             MLflow versions to @scorer functions)
#   expectations → per-row guidelines for LLM-judge scorers

# Dataset structure:
#   inputs      → kwargs passed to predict_fn; expected_agent drives routing_accuracy
#   guidelines  → per-row guidelines for ExpectationsGuidelines (top-level, as MLflow 3.x expects)
#   expectations → same guidelines nested for older MLflow versions (belt-and-suspenders)

EVAL_DATASET = [
    # ── Revenue domain ────────────────────────────────────────────────────────
    {
        "inputs": {"question": "Which location has the highest order cancellation rate right now?", "expected_agent": "revenue"},
        "guidelines": [
            "Must name exactly one specific location as having the highest rate — not a list.",
            "Must include a numeric cancellation rate (percentage).",
            "Must not say it cannot access the data or ask for clarification.",
        ],
        "expectations": {"expected_agent": "revenue"},
    },
    {
        "inputs": {"question": "How does revenue compare across our locations this week?", "expected_agent": "revenue"},
        "guidelines": [
            "Must provide revenue figures or a ranking for multiple locations.",
            "Must reference the current week as the time period.",
            "Must not return a generic description without actual numbers.",
        ],
        "expectations": {"expected_agent": "revenue"},
    },
    {
        "inputs": {"question": "Which brand is generating the most revenue right now?", "expected_agent": "revenue"},
        "guidelines": [
            "Must name a specific brand (not a location).",
            "Must include a revenue figure or ranking.",
            "Must distinguish between brand-level and location-level performance.",
        ],
        "expectations": {"expected_agent": "revenue"},
    },
    # ── Operations domain ─────────────────────────────────────────────────────
    {
        "inputs": {"question": "Which location needs the most operational attention right now?", "expected_agent": "operations"},
        "guidelines": [
            "Must name one specific location with the highest risk.",
            "Must justify with at least two operational metrics (e.g. complaint rate, cancel rate, food safety).",
            "Must not give a generic answer — must reference actual data from the system.",
        ],
        "expectations": {"expected_agent": "operations"},
    },
    # ── Inspection domain (document RAG) ──────────────────────────────────────
    {
        "inputs": {"question": "What happened during the Chicago food safety inspection? Were there critical violations?", "expected_agent": "inspection"},
        "guidelines": [
            "Must reference a specific inspection report by date or ID.",
            "Must state the inspection score and/or grade.",
            "Must explicitly state whether critical violations were found.",
            "Must cite at least one specific violation if they exist.",
        ],
        "expectations": {"expected_agent": "inspection"},
    },
    # ── Legal domain (document RAG) ───────────────────────────────────────────
    {
        "inputs": {"question": "Do we have any active high-risk legal cases? What is the total financial exposure?", "expected_agent": "legal"},
        "guidelines": [
            "Must confirm whether active cases exist — cannot hedge or say data is unavailable.",
            "Must cite at least one specific case number (format: CK-XX-XXXX).",
            "Must include a risk classification (HIGH / MEDIUM / LOW) for at least one case.",
            "Must state a financial exposure amount.",
            "Must include a reminder to involve legal counsel.",
        ],
        "expectations": {"expected_agent": "legal"},
    },
    {
        "inputs": {"question": "Which location has the most active legal cases?", "expected_agent": "legal"},
        "guidelines": [
            "Must name a specific location — not \'I don\'t know\' or a list of all locations.",
            "Must provide a count of active cases for at least the top location.",
            "Must distinguish active cases from settled or dismissed ones.",
        ],
        "expectations": {"expected_agent": "legal"},
    },
    # ── Regulatory domain (document RAG) ──────────────────────────────────────
    {
        "inputs": {"question": "Are there any permits or regulatory certificates expiring in the next 60 days?", "expected_agent": "regulatory"},
        "guidelines": [
            "Must list specific document IDs or permit names — not generic descriptions.",
            "Must include expiry dates for each flagged document.",
            "Must specify which location each document applies to.",
            "If nothing is expiring, must say so explicitly with supporting evidence.",
        ],
        "expectations": {"expected_agent": "regulatory"},
    },
    # ── Consultancy domain (document RAG) ─────────────────────────────────────
    {
        "inputs": {"question": "What do our consultants recommend as the top AI investments for the next 90 days?", "expected_agent": "consultancy"},
        "guidelines": [
            "Must reference a specific consulting report or firm by name.",
            "Must include at least one concrete recommendation (not just \'invest in AI\').",
            "Must include a financial metric — ROI estimate, cost, or projected saving.",
            "Must frame the answer in the 90-day horizon.",
        ],
        "expectations": {"expected_agent": "consultancy"},
    },
    # ── Audit domain (document RAG) ───────────────────────────────────────────
    {
        "inputs": {"question": "What were the most significant audit findings this quarter?", "expected_agent": "audits"},
        "guidelines": [
            "Must cite the auditing firm and the audit period covered.",
            "Must classify findings by severity (Critical / Significant / Minor / Informational).",
            "Must state remediation status for at least one finding.",
            "Must not conflate audit findings with food safety inspection findings.",
        ],
        "expectations": {"expected_agent": "audits"},
    },
    # ── Cross-domain synthesis ─────────────────────────────────────────────────
    {
        "inputs": {"question": "Give me a board deck summary: revenue performance, top operational risk, legal exposure, and one strategic recommendation.", "expected_agent": "multi"},
        "guidelines": [
            "Must explicitly address all four domains: revenue, operations, legal, and strategy.",
            "Must include at least one concrete number or data point per domain.",
            "Must not refuse or hedge on any of the four domains.",
            "Response must be structured — not a single unbroken paragraph.",
        ],
        "expectations": {"expected_agent": "multi"},
    },
]

df = pd.DataFrame(EVAL_DATASET)
print(f"Evaluation dataset: {len(df)} questions across {df['expectations'].apply(lambda x: x['expected_agent']).nunique()} agent domains")
df[["inputs", "guidelines"]]


## 3. Define Scorers

| Scorer | Type | What it checks |
|---|---|---|
| `ExpectationsGuidelines()` | Built-in LLM judge | Per-row pass/fail criteria from `expectations.guidelines` |
| `RelevanceToQuery()` | Built-in LLM judge | Is the response directly relevant and responsive to the question asked? |
| `Safety()` | Built-in LLM judge | Does the response contain harmful or inappropriate content? |
| `routing_accuracy` | Custom `@scorer` | Does the response vocabulary match the expected sub-agent domain? |
| `cites_specific_data` | Custom `@scorer` | Does the response cite numbers/dates/IDs rather than hedge? |

> **`ExpectationsGuidelines` vs `Guidelines`:** `Guidelines()` applies the *same* criteria to every row — useful for global rules like "always respond in English". `ExpectationsGuidelines()` reads `guidelines` from each row's `expectations` dict, so each question gets its own pass/fail criteria. Since every question here has different requirements, `ExpectationsGuidelines` is correct.
>
> **`RelevanceToQuery`:** LLM judge that evaluates whether the response actually answers the question asked — catches cases where the supervisor returns a response that passes safety and guideline checks but doesn't address what the CEO asked.
>
> **No `Correctness` scorer.** Correctness compares against a hardcoded `expected_response`. For a live analytics system the data changes constantly — hardcoded gold answers become false failures after every simulator update. `ExpectationsGuidelines` + `cites_specific_data` cover the same intent without brittleness.


In [ ]:
import re
try:
    from mlflow.genai.scorers import scorer, ExpectationsGuidelines, RelevanceToQuery, Safety, Guidelines
    _has_relevance = True
except ImportError:
    from mlflow.genai.scorers import scorer, ExpectationsGuidelines, Safety, Guidelines
    RelevanceToQuery = None
    _has_relevance = False
    print("⚠️  RelevanceToQuery not available in this MLflow version — skipping")
from mlflow.entities import Feedback

# ── Domain vocabulary signals ─────────────────────────────────────────────────
# Each sub-agent produces responses with characteristic vocabulary.
# We use this to infer routing without access to internal MAS trace data.
_AGENT_SIGNALS = {
    "revenue": ["revenue", "cancellation rate", "order count", "orders placed", "sales", "avg order", "weekly", "total orders"],
    "operations": ["complaint rate", "kitchen", "operational", "throughput", "busiest", "food safety grade", "cancel rate"],
    "inspection": ["inspection report", "violation", "inspector", "corrective action", "grade", "score", "health permit"],
    "legal": ["case no", "ck-", "risk level", "amount at stake", "counsel", "litigation", "plaintiff", "high risk"],
    "regulatory": ["permit", "certificate", "expiry", "regulatory", "fda", "zoning", "issuing authority"],
    "audits": ["audit", "auditor", "finding", "pwc", "deloitte", "kpmg", "significant", "critical finding"],
    "consultancy": ["consultant", "roi", "recommend", "strategy", "mckinsey", "phase 1", "investment"],
    "multi": ["revenue", "legal", "audit", "operational"],  # board summary — needs all domains
}


@scorer
def routing_accuracy(inputs: dict, outputs: str, expectations: dict = None) -> Feedback:
    """
    Checks whether the MAS response contains vocabulary signals consistent
    with the expected sub-agent domain for this question.
    Returns a score from 0.0 (wrong domain) to 1.0 (clearly routed correctly).
    """
    expected_agent = (inputs or {}).get("expected_agent") or (expectations or {}).get("expected_agent", "")
    response_lower = (outputs or "").lower()

    if not expected_agent or expected_agent not in _AGENT_SIGNALS:
        return Feedback(value=None, rationale="No expected_agent specified — skipped")

    signals = _AGENT_SIGNALS[expected_agent]

    if expected_agent == "multi":
        domains_present = sum(
            1 for domain_signals in _AGENT_SIGNALS.values()
            if any(s in response_lower for s in domain_signals)
        )
        score = min(1.0, domains_present / 4.0)
        rationale = f"{domains_present} agent domains detected in board summary response"
    else:
        matched = [s for s in signals if s in response_lower]
        score = min(1.0, len(matched) / max(2, len(signals) // 2))
        rationale = f"Matched signals: {matched}" if matched else "No expected-agent signals detected in response"

    return Feedback(value=score, rationale=rationale)


@scorer
def cites_specific_data(inputs: dict, outputs: str, expectations: dict = None) -> Feedback:
    """
    Checks whether the response cites concrete data points rather than
    giving vague or hedged answers. CEO responses must have numbers.
    """
    response = outputs or ""

    patterns = [
        r'\d+\.?\d*\s*%',                        # percentages
        r'\$\s*[\d,]+',                           # dollar amounts
        r'CK-\d+-\d+',                            # legal case numbers
        r'\d{4}-\d{2}-\d{2}',                     # ISO dates
        r'\b(?:score|grade)\s*(?:of\s*)?\d+',     # inspection scores
        r'\b\d+x\b',                              # ROI multiples (e.g. 3.2x)
        r'\b\d+\s+(?:cases?|orders?|locations?|findings?)',  # count phrases
    ]

    found = [pat for pat in patterns if re.search(pat, response, re.IGNORECASE)]

    hedges = ["i don't have access", "i cannot provide", "i'm unable", "consult your", "please check with"]
    is_hedged = any(h in response.lower() for h in hedges)

    if is_hedged and len(found) < 2:
        return Feedback(value=0.0, rationale="Response is hedged with no concrete data — agent likely failed to retrieve")

    score = min(1.0, len(found) / 3.0)
    return Feedback(value=score, rationale=f"Data patterns found: {len(found)} types — {found}")


print("Scorers defined:")
print("  - ExpectationsGuidelines  (LLM judge: per-row pass/fail from expectations.guidelines)")
print("  - RelevanceToQuery        (LLM judge: is the response directly relevant to the question?)")
print("  - Safety                  (LLM judge: harmful/inappropriate content check)")
print("  - Guidelines              (LLM judge: global guideline for trace-derived eval — trace eval only)")
print("  - routing_accuracy        (custom: did MAS route to the right domain?)")
print("  - cites_specific_data     (custom: does the answer contain concrete numbers?)")


## 4. Run Evaluation

`mlflow.genai.evaluate()` calls `predict_fn` for each row, applies all scorers, and logs everything to the MLflow experiment as a named run.

> This will make **10 calls** to the CEO Supervisor endpoint. Each call takes ~30–120 seconds depending on agent routing. Expect total runtime of **5–15 minutes**.

In [ ]:
# Log the evaluation dataset to MLflow so it's visible in the Experiments UI.
# This makes the dataset a first-class artifact linked to the run — reviewers
# can see exactly which questions were used for evaluation without reading code.
_eval_df = pd.DataFrame([
    {
        "question":       row["inputs"]["question"],
        "expected_agent": row["inputs"]["expected_agent"],
        "guidelines":     "\n".join(row["guidelines"]),
    }
    for row in EVAL_DATASET
])
_mlflow_dataset = mlflow.data.from_pandas(
    _eval_df,
    source="CEO Evaluation — Curated Dataset",
    name="ceo_supervisor_eval_dataset",
)

# _mas_id is defined in cell 3 (e.g. "mas-e693df14")
_eval_run_name = f"{_mas_id}-dev-experiment"

with mlflow.start_run(run_name=_eval_run_name):
    mlflow.log_input(_mlflow_dataset, context="eval")
    results = mlflow.genai.evaluate(
        predict_fn=predict_fn,
        data=EVAL_DATASET,
        scorers=[
            ExpectationsGuidelines(),
            *([RelevanceToQuery()] if _has_relevance else []),
            Safety(),
            routing_accuracy,
            cites_specific_data,
        ],
    )

print("\n✅ Evaluation complete")
# Diagnostics: inspect the results object so we know its shape
print(f"results type: {type(results)}")
print(f"results.metrics: {getattr(results, 'metrics', 'N/A')}")
_tables = getattr(results, 'tables', None)
if _tables is not None:
    if isinstance(_tables, dict):
        print(f"results.tables keys: {list(_tables.keys())}")
        for _k, _v in _tables.items():
            try:
                print(f"  [{_k!r}] shape={_v.shape}, cols={_v.columns.tolist()}")
            except Exception:
                print(f"  [{_k!r}] type={type(_v)}")
    else:
        print(f"results.tables type: {type(_tables)}, value: {_tables}")
else:
    print("results has no .tables attribute")
    print(f"results attrs: {[a for a in dir(results) if not a.startswith('_')]}")


## 5. Inspect Results

The results table shows one row per question × scorer. Use it to identify which questions the supervisor handled poorly and which agents need tuning.

In [ ]:
# Results as a pandas DataFrame
# Safely access the eval results — MLflow API shape varies by version
import pandas as _pd

results_df = _pd.DataFrame()
try:
    if hasattr(results, "tables") and isinstance(results.tables, dict):
        results_df = results.tables.get("eval_results", _pd.DataFrame())
        if results_df.empty and results.tables:
            results_df = next(iter(results.tables.values()), _pd.DataFrame())
    elif hasattr(results, "_results_df"):
        results_df = results._results_df
    elif hasattr(results, "to_pandas"):
        results_df = results.to_pandas()
except Exception as _e:
    print(f"⚠️  Could not access eval results table: {_e}")

print(f"Results shape: {results_df.shape}")
print(f"Available columns: {results_df.columns.tolist()}")

if results_df.empty:
    print("⚠️  Empty results DataFrame — evaluation may not have completed")
else:
    # Detect question column
    _q_col = next(
        (c for c in results_df.columns
         if c in ("inputs/question", "request", "question")
         or (c == "inputs" and not c.endswith("/"))),
        results_df.columns[0],
    )
    # Detect outputs column
    _out_col = next(
        (c for c in results_df.columns
         if c in ("outputs", "response", "output")),
        None,
    )
    # Scorer columns — cover both MLflow 2.x and 3.x naming conventions
    _scorer_prefixes = ["guidelines", "expectations_guidelines", "correctness",
                        "safety", "routing", "cites"]
    scorer_cols = [c for c in results_df.columns
                   if any(c.startswith(s) for s in _scorer_prefixes)]
    print(f"Question col: {_q_col!r} | Outputs col: {_out_col!r} | Scorer cols: {scorer_cols}")

    _display_cols = [_q_col] + ([_out_col] if _out_col else []) + scorer_cols
    try:
        display(
            results_df[_display_cols]
            .rename(columns={_q_col: "question"})
            .assign(question=lambda df: df["question"].astype(str).str[:80] + "...")
        )
    except Exception as _e:
        print(f"⚠️  Display failed ({_e}); showing full DataFrame:")
        display(results_df)


In [ ]:
# ── Score summary ─────────────────────────────────────────────────────────────
if not scorer_cols:
    print("⚠️  No scorer columns found — check column names above")
else:
    numeric_scorer_cols = [c for c in scorer_cols if "score" in c or "value" in c.lower()]
    if not numeric_scorer_cols:
        numeric_scorer_cols = scorer_cols

    try:
        summary = results_df[numeric_scorer_cols].apply(
            lambda col: _pd.to_numeric(col, errors="coerce")
        ).agg(["mean", "min", "max"]).T
        summary.columns = ["mean", "min", "max"]
        summary["mean"] = summary["mean"].round(3)
        print("Scorer summary (higher = better):")
        display(summary.sort_values("mean"))
    except Exception as _e:
        print(f"⚠️  Summary failed ({_e})")
        display(results_df[numeric_scorer_cols])


In [ ]:
# ── Flag failing questions ────────────────────────────────────────────────────
# Questions where the cites_specific_data score is 0 indicate the agent
# probably failed to retrieve data (hedged response with no numbers).

cites_col = next((c for c in scorer_cols if "cites" in c), None)
if cites_col:
    failing = results_df[results_df[cites_col] < 0.3][[_q_col, cites_col]]
    if len(failing):
        print(f"⚠️  {len(failing)} question(s) returned hedged/empty responses:")
        for _, row in failing.iterrows():
            _q_val = row[_q_col]
            if isinstance(_q_val, dict):
                _q_val = _q_val.get("question", str(_q_val))
            print(f"  - {str(_q_val)[:100]}")
    else:
        print("✅ All questions returned data-backed responses")

## 6. Build an Evaluation Dataset from Production Traces

In addition to a curated dataset, you can harvest **real CEO questions** from production traces and turn them into an evaluation dataset. This is the recommended workflow for continuous quality monitoring:

```
Production traffic → MLflow traces → filter interesting traces → add ground truth → eval dataset
```

The CEO Supervisor endpoint automatically writes traces to MLflow — no instrumentation code needed.

In [ ]:
# ── Find the experiment with production traces ────────────────────────────────
# Always look up by the endpoint path that Databricks auto-creates for the MAS.
# Using MAS_EXPERIMENT_ID directly breaks whenever Databricks rotates the
# experiment ID on a serving-endpoint update — which happens on every PUT.

experiment_id = None
_svc_exp_name = f"/Serving/{SUPERVISOR_ENDPOINT}"
try:
    _exp = mlflow.get_experiment_by_name(_svc_exp_name)
    if _exp:
        experiment_id = _exp.experiment_id
        print(f"Found production traces experiment: {_svc_exp_name} ({experiment_id})")
    else:
        print(f"\u26a0\ufe0f  No experiment found at '{_svc_exp_name}'.")
        print("   Send a few questions through the CEO Dashboard app first, then re-run.")
except Exception as e:
    print(f"Could not find endpoint experiment: {e}")


In [ ]:
if experiment_id:
    # ── Fetch recent traces ───────────────────────────────────────────────────
    # Use the MLflow search_traces API to pull recent successful traces.
    # Filter: only OK status (skip errors), limit to last 50 traces.

    traces = mlflow.search_traces(
        experiment_ids=[experiment_id],
        filter_string="status = 'OK'",
        max_results=50,
        order_by=["timestamp DESC"],
    )

    print(f"Found {len(traces)} traces from the CEO Supervisor endpoint")

    # ── Extract inputs and outputs ────────────────────────────────────────────
    trace_records = []
    for trace in traces:
        try:
            # Root span contains the full conversation input/output
            root = trace.data.spans[0]
            inputs_raw = root.inputs or {}
            outputs_raw = root.outputs or {}

            # Extract the user's question
            messages = inputs_raw.get("messages", [])
            question = next(
                (m["content"] for m in messages if m.get("role") == "user"),
                None
            )

            # Extract the assistant's response (handles both Chat Completions and Responses API)
            response = _extract_text(outputs_raw)

            if question and response:
                trace_records.append({
                    "trace_id": trace.info.request_id,
                    "timestamp": trace.info.timestamp_ms,
                    "latency_ms": trace.info.execution_time_ms,
                    "question": question,
                    "response": response[:500],
                })
        except Exception as _te:
            print(f"  \u26a0\ufe0f  Skipped trace {getattr(trace.info, 'request_id', '?')}: {_te}")
            continue

    traces_df = pd.DataFrame(trace_records)
    print(f"\nExtracted {len(traces_df)} valid question/response pairs")
    if not traces_df.empty:
        display(traces_df[["question", "latency_ms", "response"]].head(10))
    else:
        print("No valid traces extracted.")
else:
    print("Skipping trace fetch — no experiment found yet.")
    traces_df = pd.DataFrame()

In [ ]:
trace_eval_data = []

if not traces_df.empty:
    # ── Identify slow traces ──────────────────────────────────────────────────
    # Questions that took > 60s may indicate routing to a cold endpoint or a
    # hard question that caused agent confusion.
    slow_threshold_ms = 60_000
    slow = traces_df[traces_df["latency_ms"] > slow_threshold_ms].copy()
    print(f"Slow traces (>{slow_threshold_ms/1000:.0f}s): {len(slow)}")
    if not slow.empty:
        display(slow[["question", "latency_ms"]].sort_values("latency_ms", ascending=False))

    # ── Build evaluation dataset from traces ──────────────────────────────────
    # Real CEO questions — no per-row guidelines.
    # A global guideline is passed to Guidelines() at eval time (cell below).
    trace_eval_data = [
        {"inputs": {"question": row["question"]}}
        for _, row in traces_df.head(20).iterrows()
    ]

    print(f"\nTrace-derived evaluation dataset: {len(trace_eval_data)} records")
    print("Sample question:", trace_eval_data[0]["inputs"]["question"][:100])


In [ ]:
# Global quality bar applied to every trace-derived question.
# Guidelines() applies the same criteria to all rows — correct for
# unsupervised traffic where we have no per-question expected behaviour.
_TRACE_GUIDELINE = (
    "Response must include specific data points (numbers, locations, dates, or names). "
    "Response must not be a generic hedge or refusal. "
    "Response must be directly relevant to the question asked."
)

if not traces_df.empty and trace_eval_data:
    with mlflow.start_run(run_name=f"{_mas_id}-dev-trace-experiment"):
        trace_results = mlflow.genai.evaluate(
            predict_fn=predict_fn,
            data=trace_eval_data,
            scorers=[
                Guidelines(guidelines=_TRACE_GUIDELINE),
                *([RelevanceToQuery()] if _has_relevance else []),
                Safety(),
                routing_accuracy,
                cites_specific_data,
            ],
        )

    print("\u2705 Trace-based evaluation complete")
    try:
        _tr_df = trace_results.tables.get("eval_results") if isinstance(
            getattr(trace_results, "tables", None), dict
        ) else None
        display(_tr_df if _tr_df is not None else trace_results)
    except Exception as _e:
        print(f"\u26a0\ufe0f  Could not display trace results ({_e})")
else:
    print("Skipping trace evaluation — no trace data available.")


## 7. Comparing Runs

Use the MLflow Experiments UI to compare runs. The two runs created by this notebook are:

| Run | Dataset | What it tells you |
|---|---|---|
| `ceo-supervisor-full-eval` | 10 curated Q&A pairs with ground truth | How well the supervisor handles the demo script questions |
| `ceo-supervisor-trace-eval` | Real CEO questions from production traces | General quality over real traffic |

### How to interpret the scorer results

| Score | Meaning | Action |
|---|---|---|
| `guidelines` < 0.5 | Answer doesn't satisfy the stated criteria | Improve the sub-agent's `instructions` in `ceo_knowledge_agents.ipynb` or the supervisor's `instructions` in `ceo_supervisor.ipynb` |
| `correctness` < 0.5 | Factually wrong or hallucinated answer | Check the document corpus in the UC Volume — documents may be missing or outdated |
| `routing_accuracy` < 0.5 | MAS routed to the wrong agent | Improve the `description` field for that sub-agent in `ceo_supervisor.ipynb` |
| `cites_specific_data` = 0.0 | Hedged/empty response | Sub-agent had a retrieval failure — check endpoint status and UC Volume contents |
| `safety` < 1.0 | Potentially harmful content in response | Review the full response — this is rare but warrants investigation |

### Finding low-quality traces to fix

```python
# Find all traces where the agent failed to cite specific data
mlflow.search_traces(
    experiment_ids=[experiment_id],
    filter_string="status = 'OK'",
    max_results=100,
)
# Then tag the failing ones for your eval dataset:
# mlflow.update_trace(request_id=trace_id, tags={"quality": "low", "issue": "no-data"})
```

### Next steps

1. **MemAlign** — after running this notebook, use `mlflow.genai.align()` to calibrate the `Guidelines` judge to match your specific definition of a good CEO response
2. **Production monitoring** — link the MAS experiment to a Unity Catalog table and set up a scheduled evaluation job to track quality over time
3. **Prompt optimization** — use `mlflow.genai.optimize_prompts()` with the evaluation dataset to automatically improve the supervisor's `instructions` based on failing scores

In [ ]:
# ── Quick link to MLflow Experiments UI ───────────────────────────────────────
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
host = ctx.apiUrl().get()
try:
    _exp = mlflow.get_experiment_by_name(EVAL_EXPERIMENT_NAME)
    exp_id = _exp.experiment_id if _exp else None
except Exception:
    exp_id = None
if exp_id:
    print(f"View results in MLflow:")
    print(f"   {host}/ml/experiments/{exp_id}")